# Oppitunti 12 - Keskusteluhistorian Tiivistäminen Agentin Muistiinpanolla

Tämä muistikirja näyttää, kuinka hallita kontekstia pitkissä keskusteluissa Microsoft Agent Frameworkin avulla. Keskustelujen kasvaessa tokenien määrä lisääntyy — lopulta ylittäen mallin kontekstiikkunan koon. Ratkaisemme tämän **kontekstin tiivistämisen mallilla** ja **agentin muistiinpanolla**, joka toimii pysyvänä muistina.

## Mitä opit:
1. **Miksi kontekstinhallinta on tärkeää**: Token-rajojen ja kontekstiikkunoiden ymmärtäminen  
2. **Kontekstitietoiset agentit**: Agenttien rakentaminen, jotka hallitsevat omaa keskustelukontekstiaan  
3. **Kontekstin tiivistämisen malli**: Työkalujen käyttö keskusteluhistorian tiivistämiseen  
4. **Agentin muistiinpano**: Pysyvä muisti, joka säilyy kontekstin tiivistämisen yli

## Edellytykset:
- Azure OpenAI -ympäristön asennus ja ympäristömuuttujien määrittäminen  
- Peruskäsitys agenteista aiemmista oppitunneista


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miksi kontekstinhallinta on tärkeää

Jokaisella LLM:llä on rajallinen **kontekstikehys** — suurin määrä tokeneita, jonka se voi käsitellä yhdessä pyynnössä. Monivaiheisen keskustelun aikana:

- **Tokenien määrä kasvaa lineaarisesti** jokaisen käyttäjän viestin ja avustajan vastauksen myötä.
- **Prompt-tokenit hallitsevat kustannuksia** koska koko historian lähetys toistuu jokaisella kierroksella.
- Lopulta keskustelu **ylittää kontekstikehyksen** ja malli joko lyhentää tai antaa virheen.

### Strategiat kontekstin hallintaan

| Strategia | Toimintatapa | Kompromissi |
|---|---|---|
| **Lyhentäminen** | Vanhemmat viestit pudotetaan pois | Varhaisen kontekstin menetys |
| **Tiivistäminen** | Vanhemmat viestit tiivistetään yhteenvedoksi | Jotain yksityiskohtia menetetään, mutta avainkohdat säilyvät |
| **Muistilehtiö / Ulkoinen muisti** | Tärkeät tiedot tallennetaan keskustelun ulkopuolelle | Vaatii työkalukutsuja, mutta säilyy kaikenlaisissa lyhennyksissä |

Tässä muistikirjassa yhdistämme **tiivistämisen** ja **muistilehtiötyökalun**, jotta agentti voi ylläpitää jatkuvuutta vaikka keskusteluhistoria tiivistetään.


## Kontekstia Tietoinen Agentin Luominen


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pitkän keskustelun simulointi

Käydään läpi monikierroksinen keskustelu nähdäksesi, miten konteksti kertyy. Agentin tulisi säilyttää keskeiset tiedot (mieltymykset, budjetti, matkustuspäivät) kierrosten välillä ja osoittaa jatkuvuutta.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Huomaa, miten agentti säilyttää kontekstin aiemmista vuoroista — se tietää Japanista, sushista, temppeleistä, valokuvauksesta, 3000 dollarin budjetista, yksin matkustamisesta ja huhtikuun aikavälistä. Lyhyessä keskustelussa tämä toimii hyvin, mutta keskustelun kasvaessa koko historia on kallista lähettää uudelleen.

Jatketaan keskustelua useammilla vuoroilla nähdäksesi kontekstin kertymistä:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontekstin tiivistämismalli

Keskustelun edetessä voimme käyttää **tiivistystyökalua** tiivistämään kertynyttä kontekstia tiiviiseen muotoon. Agentti kutsuu tätä työkalua tallentaakseen keskeiset mieltymykset, jotta olennaiset tiedot säilyvät, vaikka vanhemmat viestit poistettaisiin.

Tämä malli on rakennusosa monimutkaisemmille historian tiivistämisille:
1. Agentti tunnistaa keskustelusta keskeiset faktat
2. Se kutsuu tiivistystyökalua tallentaakseen ne
3. Vanhemmat viestit voidaan turvallisesti poistaa, koska tiivistelmä kattaa olennaisen

Alla määrittelemme `summarize_preferences`-työkalun, jota agentti voi kutsua tallentaakseen tiiviin yhteenvedon oppimistaan asioista.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Yhteenveto

Tässä oppitunnissa opit hallitsemaan kontekstia pitkäkestoisissa agenttikeskusteluissa Microsoft Agent Frameworkin avulla:

### Keskeiset käsitteet
- **Kontekstin ikkunat ovat rajallisia** — jokainen keskusteluhistorian token maksaa ja lasketaan rajoitusta vastaan.
- **Yhteenvedon teko työkalut** antavat agentille mahdollisuuden tiivistää kertyneestä kontekstista yhteenvetoja, jotka vähentävät tokenien käyttöä säilyttäen olennaiset tiedot.
- **Agentin muistiinpanot** tarjoavat pysyvän ulkoisen muistin, joka säilyy keskustelun supistuksista huolimatta.

### Mitä rakensit
- **Kontekstia tunnistava agentti**, joka ylläpitää jatkuvuutta monikierroksisissa keskusteluissa
- **Yhteenvedon työkalu** (`summarize_preferences`), joka tallentaa käyttäjän tärkeimmät tiedot tiiviissä muodossa
- **Monikierroksinen keskustelu**, joka havainnollistaa kontekstin säilyttämistä ja muutosten käsittelyä

### Käytännön sovellukset
- **Asiakaspalvelubotit**: Muistaa mieltymykset pitkien tukisessioiden aikana
- **Henkilökohtaiset avustajat**: Seuraa käynnissä olevia projekteja ilman kontekstin uudelleen selittämistä
- **Opettajat ja tutorit**: Ylläpitää oppilaan edistymistä monien vuorovaikutusten aikana

### Seuraavat askeleet
- Toteuta täydellinen muistiinpanotyökalu tiedostopohjaisella pysyvyydellä
- Lisää automaattinen historian lyhentäminen yhteenvedon jälkeen
- Yhdistä vektoritietokantoihin semanttisen muistin hakua varten
- Rakenna agenteja, jotka voivat jatkaa keskusteluja päiviä myöhemmin täydellä kontekstilla


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, huomioithan, että automaattiset käännökset voivat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä tulisi pitää virallisena lähteenä. Tärkeissä tiedoissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä johtuvista väärinymmärryksistä tai virhetulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
